### Installation

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
%%capture
import os, re
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    !pip install --upgrade -qqq uv
    !uv pip install vllm==0.11.2 unsloth-zoo unsloth
    !uv pip install transformers==4.56.2
    !uv pip install --no-deps trl==0.22.2
!pip install func_timeout
!pip install rapidfuzz sqlglot
!pip install wandb

### Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 10240 # 5500 input, 4096 output

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    gpu_memory_utilization = 0.90, # Don't need to worry about this with UNSLOTH STANDBY
    dtype = torch.float16
)
FastLanguageModel.for_inference(model)

# Load Dataset

In [ ]:
DB_ROOT_PATH = 'path/to/dev_corrected/dev_databases'

In [ ]:
from datasets import load_dataset
from huggingface_hub import login
from huggingface_hub import hf_hub_download
import json
repo_id = "RuihanCao/bird-dev-corrected"

# 1. Load the Indexed Dataset
dataset_path = hf_hub_download(repo_id=repo_id, filename="dev_dataset_indexed.json", repo_type="dataset")
with open(dataset_path, 'r') as f:
    eval_dataset = json.load(f)

# 2. Load the Gold Results (for comparison during inference)
gold_path = hf_hub_download(repo_id=repo_id, filename="gold_results.json", repo_type="dataset")
with open(gold_path, 'r') as f:
    gold_results = json.load(f)

# 3. Load the Schema Cache (for prompt generation)
schema_path = hf_hub_download(repo_id=repo_id, filename="bird_schema_cache.json", repo_type="dataset")
with open(schema_path, 'r') as f:
    schema_cache = json.load(f)

print(f"✅ Loaded {len(eval_dataset)} entries and {len(schema_cache)} cached schemas.")

In [ ]:
print(eval_dataset[0])
print(gold_results['0'])
print(schema_cache['california_schools'])

# Helper Functions

In [ ]:
import time
from func_timeout import func_timeout, FunctionTimedOut

def _run_query(db_path, sql):
      conn = sqlite3.connect(db_path)
      cursor = conn.cursor()
      cursor.execute(sql)
      result = cursor.fetchall()
      conn.close()
      return set(result)

def execute_sql(db_path, sql):
    try:
        # 5 sec limit
        return func_timeout(5, _run_query, args=(db_path, sql))
    except FunctionTimedOut:
        return "Error: Timeout (SQL took too long)"
    except Exception as e:
        return f"Error: {e}"

def clean_sql(text, word_limit=375):
    # Remove the <think> block
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    # Remove <answer> tags
    text = re.sub(r'<answer>', '', text, flags=re.DOTALL)
    text = re.sub(r'</answer>', '', text, flags=re.DOTALL)
    # Extract code block if it exists (Standard Instruct format)
    # Looks for ```sql ... ``` or just ``` ... ```
    code_block = re.search(r'```(?:sql)?\s*(.*?)\s*```', text, re.DOTALL | re.IGNORECASE)
    if code_block:
        return code_block.group(1).strip()

    # Fallback
    extracted = text.strip()
    words = extracted.split()
    if len(words) > word_limit:
        #print(f"⚠️ Extraction Warning: SQL candidate too long ({len(words)} words). Truncating to last {word_limit} words.")
        # Take the last 375 words
        extracted = " ".join(words[-word_limit:])
    return extracted

def format_result_log(res, max_len=100):
    if isinstance(res, str):
        return res # Return full error message

    res_str = str(res)
    if len(res_str) > max_len:
        return res_str[:max_len] + "... [TRUNCATED]"
    return res_str

import sqlite3
import sqlglot
from sqlglot import exp
from rapidfuzz import process, fuzz
import os

def fix_sql_values_fuzzy(db_path, sql_query, threshold=85):
    """
    Parses SQL, extracts string literals in WHERE/HAVING clauses,
    and fuzzy matches them against the database content to fix typos/case issues.
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database not found at: {db_path}")

    # Connect to DB
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Parse SQL
    try:
        parsed = sqlglot.parse_one(sql_query, read="sqlite")
    except Exception as e:
        print(f"Error parsing SQL: {e}")
        return sql_query  # Return original if parse fails

    # Identify tables in the query to limit search scope
    tables = [t.name for t in parsed.find_all(exp.Table)]

    # Extract binary operations involving strings (e.g., col = 'string', col LIKE 'string')
    # We look for Equality (EQ) or Like (LIKE) nodes where one side is a Literal String
    modifications = {}

    for node in parsed.find_all(exp.EQ):
        left = node.left
        right = node.right

        # Identify which side is the column and which is the string literal
        col_node = None
        val_node = None

        if isinstance(left, exp.Column) and isinstance(right, exp.Literal) and right.is_string:
            col_node = left
            val_node = right
        elif isinstance(right, exp.Column) and isinstance(left, exp.Literal) and left.is_string:
            col_node = right
            val_node = left

        if col_node and val_node:
            original_value = val_node.this
            column_name = col_node.name

            # 1. Try to find the exact column in the DB to fetch valid values
            # (Simple resolution: check if column exists in any of the active tables)
            candidate_values = []
            for table in tables:
                try:
                    # Check if column exists in this table
                    cursor.execute(f"PRAGMA table_info({table})")
                    columns_info = cursor.fetchall() # list of tuples
                    col_names = [info[1] for info in columns_info]

                    if column_name in col_names:
                        # Fetch distinct values from this column
                        # LIMITing to avoid massive overhead on huge tables
                        query = f"SELECT DISTINCT {column_name} FROM {table} WHERE {column_name} IS NOT NULL LIMIT 2000"
                        cursor.execute(query)
                        results = cursor.fetchall()
                        candidate_values.extend([str(r[0]) for r in results])
                except Exception as e:
                    continue # Table might be aliased or complex, skip safe

            if not candidate_values:
                continue # Couldn't find valid values for this column, skip

            # 2. Perform Fuzzy Matching
            # extractOne returns (best_match, score, index)
            best_match = process.extractOne(
                original_value,
                candidate_values,
                scorer=fuzz.ratio, # Use ratio for strictness, or token_sort_ratio for flexibility
                score_cutoff=threshold
            )

            if best_match:
                corrected_value, score, _ = best_match
                if corrected_value != original_value:
                    # Store modification to apply later (or modify in place)
                    # We modify in place using sqlglot
                    print(f"Fixing: '{original_value}' -> '{corrected_value}' (Score: {score})")
                    val_node.set("this", corrected_value)

    conn.close()

    # Return the reconstructed SQL
    return parsed.sql(dialect="sqlite")


from dataclasses import dataclass
import math
from typing import Any, Dict, List

@dataclass(frozen=True)
class ComparableTuple:
    """
    A tuple wrapper that handles float comparison with tolerance (1e-9).
    Crucial for voting on results where 10.0000001 must equal 10.0.
    """
    data: tuple

    def __eq__(self, other):
        if not isinstance(other, ComparableTuple):
            return False
        if len(self.data) != len(other.data):
            return False

        for v1, v2 in zip(self.data, other.data):
            if isinstance(v1, float) and isinstance(v2, float):
                if not math.isclose(v1, v2, rel_tol=1e-9):
                    return False
            elif v1 != v2:
                return False
        return True

    def __hash__(self):
        # Convert floats to strings with limited precision for hashing compatibility
        processed = tuple(f"{x:.8f}" if isinstance(x, float) else x for x in self.data)
        return hash(processed)

def execution_match(prediction: Any, ground_truth: Any) -> bool:
    """
    Compare two execution results for equality, handling float values and row order.
    Adapted to handle both List[Dict] (original code) and List[Tuple] (SQLite output).
    """
    # 1. Handle Error Strings immediately
    if isinstance(prediction, str) or isinstance(ground_truth, str):
        return prediction == ground_truth

    # 2. Handle None/Empty
    if not prediction and not ground_truth: return True
    if not prediction or not ground_truth: return False

    def transform_results(ex_result):
        new_ex_result = []
        for row in ex_result:
            # ADAPTER: Handle SQLite tuples vs Dicts
            if isinstance(row, dict):
                row_vals = tuple(row.values())
            else:
                row_vals = tuple(row)

            new_row = ComparableTuple(row_vals)
            new_ex_result.append(new_row)
        return set(new_ex_result)

    return transform_results(ground_truth) == transform_results(prediction)

def get_votes(candidate_predictions: List[Dict]) -> Dict:
    """
    Count votes for each unique prediction based on execution results.
    Clustering Algorithm: O(N^2) comparison to group semantically identical results.
    """
    clusters = [] # List of tuples: (representative_result, vote_count, [candidate_dicts])

    for candidate in candidate_predictions:
        res = candidate["pred_result"]
        found = False

        # Try to find an existing cluster that matches this result
        for i, (cluster_res, count, members) in enumerate(clusters):
            if execution_match(res, cluster_res):
                clusters[i] = (cluster_res, count + 1, members + [candidate])
                found = True
                break

        # If no match, start a new cluster
        if not found:
            clusters.append((res, 1, [candidate]))
    # Sort clusters by vote count descending
    clusters.sort(key=lambda x: x[1], reverse=True)
    # The winner is the first candidate of the largest cluster
    winner = clusters[0][2][0]
    # Summary of voting for logs
    vote_summary = f"Votes: {', '.join([str(c[1]) for c in clusters])}"
    return {
        "winner": winner,
        "vote_summary": vote_summary
    }

# Generate Prompt Function

In [ ]:
def omni_style_prompt(sample):
    db_id = sample['db_id']
    schema = schema_cache.get(db_id, "")
    evidence = sample['evidence']
    question = sample['question']
    user_prompt = f"""Task Overview:
You are a data science expert. Below, you are provided with a database schema and a natural language question. Your task is to understand the schema and generate a valid SQL query to answer the question.

Database Engine:
SQLite

Database Schema:
{schema}
This schema describes the database's structure, including tables, columns, primary keys, foreign keys, and any relevant relationships or constraints.

Question:
{evidence}
{question}

Instructions:
- Always use * 1.0 or CAST(... AS FLOAT) when dividing two integers to avoid zero-truncation (e.g., SUM(A)*1.0/SUM(B)).
- TEXT '1' is not equal to INTEGER 1, you must use CAST(... AS TYPE) when needed for all joins and numeric comparisons.
- Add IS NOT NULL for columns involved in ordering, ranking or math.
- Use RANK() for "Top" questions unless a unique sequential ID is specifically requested; prioritize ORDER BY ... DESC LIMIT X for "Top X" questions.
- Never rely on lexicographical (A-Z) sorting for quantitative data stored as TEXT. Always check the schema for a parallel numeric column. If none exists, you must CAST or parse the text column into a number before sorting.
- Use JULIANDAY for date and time arithmetic and comparisons, and strftime for formatting and extracting specific date parts for display or grouping.
- When request specifies a number of X decimal places, use the ROUND(expression, X) function.
- To accurately count occurrences in a non-normalized column (like comma-separated lists), you must split the strings.
- If the prompt asks for a "percentage, it implies a scale of 0–100 (ratio * 100).
- To get an accurate count of unique rows from a specific table after a join, use COUNT(DISTINCT primary_key_column)
- Use LEFT JOIN instead of JOIN whenever fetching optional attributes (like translations, descriptions, or notes) to ensure the primary entity is not excluded if the secondary detail is missing.
- If any selected attribute can contain multiple values inside one text field, use WITH RECURSIVE to split into one value per row.
- Always prioritize numeric/integer columns for calculations and sorting, because SQL would sort them alphabetically, making '10' smaller than '2'.
- Must arrange the columns in the SELECT clause in the exact order they are mentioned in the natural language question.
- Strictly follow the instructions in the question, and do not add filters based on common sense if they are not explicitly mentioned in the question or the evidence.
- If the question says “for each X / which X”, you must output X’s identifier even if it is not explicitly requested.
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- The generated query should return all of the information asked in the question without any missing or extra information.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
In your answer, please enclose the generated SQL query in a code block:
```
-- Your SQL query
```
"""

    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return text_prompt


In [ ]:
def self_correction_prompt(sample, sql, error):
    db_id = sample['db_id']
    schema = schema_cache.get(db_id, "")
    evidence = sample['evidence']
    question = sample['question']
    user_prompt = f"""Task Overview:
You are a data science expert. Below, you are provided with an invalid SQL query, its execution feedback, a database schema, and a natural language question. Your task is to correct the invalid SQL query, understand the schema and generate a valid SQL query to answer the question.

Database Engine:
SQLite

Database Schema:
{schema}
This schema describes the database's structure, including tables, columns, primary keys, foreign keys, and any relevant relationships or constraints.

Question:
{evidence}
{question}

Invalid SQL Query:
{sql}

Execution Feedback:
{error}

Instructions:
- Always use * 1.0 or CAST(... AS FLOAT) when dividing two integers to avoid zero-truncation (e.g., SUM(A)*1.0/SUM(B)).
- TEXT '1' is not equal to INTEGER 1, you must use CAST(... AS TYPE) when needed for all joins and numeric comparisons.
- Add IS NOT NULL for columns involved in ordering, ranking or math.
- Use RANK() for "Top" questions unless a unique sequential ID is specifically requested; prioritize ORDER BY ... DESC LIMIT X for "Top X" questions.
- Never rely on lexicographical (A-Z) sorting for quantitative data stored as TEXT. Always check the schema for a parallel numeric column. If none exists, you must CAST or parse the text column into a number before sorting.
- Use JULIANDAY for date and time arithmetic and comparisons, and strftime for formatting and extracting specific date parts for display or grouping.
- When request specifies a number of X decimal places, use the ROUND(expression, X) function.
- To accurately count occurrences in a non-normalized column (like comma-separated lists), you must split the strings.
- If the prompt asks for a "percentage, it implies a scale of 0–100 (ratio * 100).
- To get an accurate count of unique rows from a specific table after a join, use COUNT(DISTINCT primary_key_column)
- Use LEFT JOIN instead of JOIN whenever fetching optional attributes (like translations, descriptions, or notes) to ensure the primary entity is not excluded if the secondary detail is missing.
- If any selected attribute can contain multiple values inside one text field, use WITH RECURSIVE to split into one value per row.
- Always prioritize numeric/integer columns for calculations and sorting, because SQL would sort them alphabetically, making '10' smaller than '2'.
- Must arrange the columns in the SELECT clause in the exact order they are mentioned in the natural language question.
- Strictly follow the instructions in the question, and do not add filters based on common sense if they are not explicitly mentioned in the question or the evidence.
- If the question says “for each X / which X”, you must output X’s identifier even if it is not explicitly requested.
- Make sure you only output the information that is asked in the question. If the question asks for a specific column, make sure to only include that column in the SELECT clause, nothing more.
- The generated query should return all of the information asked in the question without any missing or extra information.
- Before generating the final SQL query, please think through the steps of how to write the query.

Output Format:
In your answer, please enclose the generated SQL query in a code block:
```
-- Your SQL query
```
"""

    messages = [
        {"role": "user", "content": user_prompt}
    ]

    text_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return text_prompt

# Testing

### Generation Config

In [ ]:
from vllm import SamplingParams
import time

max_seq_length = 10240

vllm_sampling_params = SamplingParams(
    n = 4,
    max_tokens = 6144,
    temperature = 0.6,
    top_p = 0.8,
    top_k = 20,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)
vllm_correction_params = SamplingParams(
    max_tokens = 6144,
    temperature = 0.6,
    top_p = 0.8,
    top_k = 20,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

### Generate All sample output

In [ ]:
START_INDEX = 42  # Start from sample 42
eval_dataset_sliced = eval_dataset[START_INDEX:]

prompts = [omni_style_prompt(sample) for sample in eval_dataset_sliced]
#test with 1 prompt
# prompts = [eval_dataset[0]['prompt']]

print(f"🚀 Starting to Generate Prompts for {len(prompts)} Questions...")
start_gen = time.time()

outputs = model.fast_generate(prompts, vllm_sampling_params)

end_gen = time.time()
avg_time_per_query = (end_gen - start_gen) / len(prompts)
print(f"✅ Inference complete. Total time: {end_gen - start_gen:.2f}s | avg time per query: {avg_time_per_query:.4f}s")

### Execute SQLs

In [ ]:
# Batch-execute SQL and verify results
from tqdm import tqdm
import json
import pandas as pd

result_save_path = f"results.csv"
print(f"🔍 Executing SQL validation... results saved to: {result_save_path}")

# Initial Executioin and track Error for self-correction
results_pass_1 = []
retry_indices = []
retry_prompts = []
for i, output_group in tqdm(enumerate(outputs), total=len(outputs), desc="Voting Pass 1"):
    global_idx = i + START_INDEX
    item = eval_dataset_sliced[i]
    db_path = os.path.join(DB_ROOT_PATH, item['db_id'], f"{item['db_id']}.sqlite")
    candidates = []
    # Process Outputs
    for output in output_group.outputs:
        raw_text = output.text
        parsed_sql = clean_sql(raw_text)
        pred_sql = fix_sql_values_fuzzy(db_path, parsed_sql)
        # Performance: Check if we've already executed this SQL for this question
        existing = next((c for c in candidates if c["pred_sql"] == pred_sql), None)
        if existing:
            candidates.append(existing.copy())
            continue
        pred_result = execute_sql(db_path, pred_sql)
        # Error Detection
        is_error = isinstance(pred_result, str)
        is_empty = (not is_error) and (len(pred_result) == 0)
        candidates.append({
          "raw_text": raw_text,
          "pred_sql": pred_sql,
          "pred_result": pred_result,
          "is_error": is_error,
          "is_empty": is_empty,
          "was_corrected": False
        })
    # Voting Logic
    # Filter out errors
    voting_pool = [c for c in candidates if not c["is_error"]]

    if not voting_pool:
        # If all 8 errored, pick the first one to correct
        winner_data = candidates[0]
        vote_msg = "All Error"
    else:
        vote_results = get_votes(voting_pool)
        winner_data = vote_results["winner"]
        vote_msg = vote_results["vote_summary"]

    # Store initial data
    results_pass_1.append({
        "global_index": global_idx,
        "item": item,
        "winner": winner_data,
        "vote_msg": vote_msg
    })
    # Determine if the WINNER needs Self-Correction
    if winner_data["is_error"] or winner_data["is_empty"]:
        if winner_data["is_empty"]:
            feedback = "The SQL executed successfully but returned an EMPTY result set."
        else:
            feedback = f"The SQL failed with the following error: {winner_data["pred_sql"]}"
        retry_indices.append(i)
        retry_prompts.append(self_correction_prompt(item, winner_data["pred_sql"], feedback))

# Batch Self-Correction (Pass 2)
if retry_prompts:
    print(f"🔄 Pass 2: Self-Correcting {len(retry_prompts)} failed winners...")
    # n=1 to save time
    corr_outputs = model.fast_generate(retry_prompts, vllm_correction_params)

    for idx, corr_output in tqdm(zip(retry_indices, corr_outputs), total=len(retry_indices), desc="Processing Corrections"):
        item = eval_dataset_sliced[idx]
        db_path = os.path.join(DB_ROOT_PATH, item['db_id'], f"{item['db_id']}.sqlite")

        # Fixed SQL and Re-execution
        corr_raw_text = corr_output.outputs[0].text
        corr_parsed_sql = clean_sql(corr_raw_text)
        corr_pred_sql = fix_sql_values_fuzzy(db_path, corr_parsed_sql)
        corr_pred_result = execute_sql(db_path, corr_pred_sql)

        # Replace the winner with the corrected version
        results_pass_1[idx]["winner"].update({
            "raw_text": f"[CORRECTED] {corr_raw_text}",
            "pred_sql": corr_pred_sql,
            "pred_result": corr_pred_result,
            "was_corrected": True
        })

# Final Logging and Metrics
final_results = []
correct_count = 0

for i, res in enumerate(results_pass_1):
    global_i = res['global_index']
    item = res['item']
    winner = res['winner']
    pred_sql = winner['pred_sql']
    pred_result = winner['pred_result']
    generated_text = winner['raw_text']
    was_corrected = winner['was_corrected']
    was_error = winner['is_error']
    was_empty = winner['is_empty']
    # Validation logic
    gold_data = gold_results.get(str(global_i), {})
    gold_json_str = gold_data.get('gold_result', "[]")
    # Decode Gold from JSON
    try:
        gold_list = json.loads(gold_json_str)
        gold_set = set(tuple(x) for x in gold_list)
    except:
        gold_set = set()

    is_correct = (pred_result == gold_set)
    if is_correct: correct_count += 1

    row_data = {
        'Index': global_i,
        "Question ID": item['question_id'],
        "Question": item['question'],
        "Gold SQL": item['SQL'],
        "Pred SQL": pred_sql,
        "Correct": is_correct,
        "Pred Res": format_result_log(pred_result),
        "Gold Res": format_result_log(gold_set),
        "Difficulty": item['difficulty'],
        "DB_ID": item['db_id'],
        "Evidence": item['evidence'],
        "Spread": res['vote_msg'],
        "Was Corrected": was_corrected,
        "Successfully Corrected": was_corrected and is_correct,
        "Corrected Error": was_error and is_correct,
        "Corrected Empty": was_empty and is_correct,
        "Gen_Text": generated_text
    }
    final_results.append(row_data)
pd.DataFrame(final_results).to_csv(result_save_path, index=False)
print(f"\n🎯 Final Accuracy (with Self-Correction): {correct_count/len(eval_dataset_sliced)*100:.2f}%")

In [ ]:
import pandas as pd

df_results = pd.read_csv(result_save_path)

# Ensure the Correct column is boolean/numeric
df_results['Correct'] = df_results['Correct'].astype(bool)

print("\n" + "="*50)
print("📊 BIRD evaluation report")
print("="*50)

# 1. Break down by difficulty
print("\n[ 1. Accuracy by difficulty level ]")
difficulty_acc = df_results.groupby('Difficulty')['Correct'].agg(['mean', 'count']).rename(columns={'mean': 'Accuracy', 'count': 'Total_Samples'})
difficulty_acc['Accuracy'] = (difficulty_acc['Accuracy'] * 100).round(2).astype(str) + '%'
print(difficulty_acc)

# 2. Break down by database (DB_ID) to find the weakest domains
print("\n[ 2. Accuracy by DB_ID (bottom 5) ]")
db_acc = df_results.groupby('DB_ID')['Correct'].agg(['mean', 'count']).rename(columns={'mean': 'Accuracy', 'count': 'Total_Samples'})
db_acc_sorted = db_acc.sort_values(by='Accuracy', ascending=True) # ascending, worst first
db_acc_sorted['Accuracy'] = (db_acc_sorted['Accuracy'] * 100).round(2).astype(str) + '%'
print(db_acc_sorted.head(5))

# 3. Totals
total_acc = df_results['Correct'].mean() * 100
print(f"\n✨ Overall accuracy: {total_acc:.2f}%")
print("="*50)

### Test Single Inference

In [ ]:
sample = eval_dataset[110]
prompts = [omni_style_prompt(sample)]

outputs = model.fast_generate(prompts, vllm_sampling_params)

raw_output = outputs[0].outputs[0].text
parsed_sql = clean_sql(raw_output)
db_path = os.path.join(DB_ROOT_PATH, sample['db_id'], f"{sample['db_id']}.sqlite")
fuzzy_matched_sql = fix_sql_values_fuzzy(db_path, parsed_sql)

print("\n" + "="*40)
print("DEBUG REPORT")
print("="*40)
print(f"Question: {sample['question']}")
print("-" * 20)
print(f"RAW MODEL OUTPUT:\n{raw_output}")
print("-" * 20)
print(f"PARSED SQL:\n{parsed_sql}")
print("-" * 20)
print(f"FUZZY MATCHED SQL:\n{fuzzy_matched_sql}")
print("-" * 20)
print(f"Gold SQL:\n{sample['SQL']}")
print("-" * 20)

pred_result = execute_sql(db_path, fuzzy_matched_sql)
is_error = isinstance(pred_result, str)
is_empty = (not is_error) and (len(pred_result) == 0)
if is_error or is_empty:
  error_type = "Execution Error" if is_error else "Empty Result Set"
  print(f"⚠️ {error_type} detected. Attempting Self-Correction...")
  if is_empty:
      feedback = "The SQL executed successfully but returned an EMPTY result set."
  else:
      feedback = f"The SQL failed with the following error: {pred_result}"
  correction_prompt = [self_correction_prompt(sample, fuzzy_matched_sql, feedback)]
  correction_outputs = model.fast_generate(correction_prompt, vllm_sampling_params)
  raw_output = correction_outputs[0].outputs[0].text
  parsed_sql = clean_sql(raw_output)
  fuzzy_matched_sql = fix_sql_values_fuzzy(db_path, parsed_sql)
  print("\n" + "="*40)
  print("CORRECTION DEBUG REPORT")
  print("-" * 20)
  print(f"Correction Raw OUTPUT:\n{raw_output}")
  print("-" * 20)
  print(f"Correction PARSED SQL:\n{parsed_sql}")
  print("-" * 20)
  print(f"Correction FUZZY MATCHED SQL:\n{fuzzy_matched_sql}")
  print("-" * 20)
  # Re-execute
  pred_result = execute_sql(db_path, fuzzy_matched_sql)

gold_json_str = sample['gold_result']
# Decode Gold from JSON
try:
    gold_list = json.loads(gold_json_str)
    gold_set = set(tuple(x) for x in gold_list)
except:
    gold_set = set()
is_correct = (pred_result == gold_set)
print(f"Execution Result (Pred): {str(pred_result)}")
print(f"Execution Result (Gold): {str(gold_set)}")

if is_correct:
    print("\n✅ VERDICT: CORRECT")
else:
    print("\n❌ VERDICT: INCORRECT")